In [1]:
import pandas as pd
import numpy as np
import json

In [2]:
with open('../data/database_202604.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

In [3]:
df = pd.DataFrame(data)
df = df.explode('value').reset_index(drop=True)
values_df = pd.json_normalize(df['value'])
df = pd.concat([df.drop('value', axis=1), values_df], axis=1)
df = df[['source_id', 'location_name', 'location_lat', 'location_lon', 'log_pm25', 'log_datetime']]
num_cols = ['location_lat', 'location_lon', 'log_pm25']
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors='coerce')
str_cols = ['source_id', 'location_name', 'log_datetime']
df[str_cols] = df[str_cols].astype('string')
df['log_datetime'] = pd.to_datetime(df['log_datetime'])
print(df.dtypes)
df

source_id        string[python]
location_name    string[python]
location_lat            float64
location_lon            float64
log_pm25                float64
log_datetime     datetime64[ns]
dtype: object


,source_id,location_name,location_lat,location_lon,log_pm25,log_datetime
0,4,ไนท์บาร์ซาร์ ต.ช้างม่อย อ.เมือง จ. เชียงใหม่,18.785477,98.999660,126.67,2026-04-01 00:00:00
1,4,ไนท์บาร์ซาร์ ต.ช้างม่อย อ.เมือง จ. เชียงใหม่,18.785477,98.999660,137.98,2026-04-01 01:00:00
2,4,ไนท์บาร์ซาร์ ต.ช้างม่อย อ.เมือง จ. เชียงใหม่,18.785477,98.999660,148.16,2026-04-01 02:00:00
3,4,ไนท์บาร์ซาร์ ต.ช้างม่อย อ.เมือง จ. เชียงใหม่,18.785477,98.999660,172.76,2026-04-01 03:00:00
4,4,ไนท์บาร์ซาร์ ต.ช้างม่อย อ.เมือง จ. เชียงใหม่,18.785477,98.999660,175.12,2026-04-01 04:00:00
...,...,...,...,...,...,...
438957,7171,วัดปงแท่น หมู่บ้านปงแท่น ต.จางเหนือ อ.แม่เมาะ ...,18.357949,99.892538,77.48,2026-04-30 19:00:00
438958,7171,วัดปงแท่น หมู่บ้านปงแท่น ต.จางเหนือ อ.แม่เมาะ ...,18.357949,99.892538,81.48,2026-04-30 20:00:00
438959,7171,วัดปงแท่น หมู่บ้านปงแท่น ต.จางเหนือ อ.แม่เมาะ ...,18.357949,99.892538,78.90,2026-04-30 21:00:00
438960,7171,วัดปงแท่น หมู่บ้านปงแท่น ต.จางเหนือ อ.แม่เมาะ ...,18.357949,99.892538,78.90,2026-04-30 22:00:00


In [4]:
print(len(df['location_name'].unique()))
normal_format = [i for i in df['location_name'].unique() if 'ต.' in i and 'อ.' in i and 'จ.' in i]
print(len(normal_format))
# print('\n'.join(normal_format))

804
577


In [5]:
df = df[df['location_name'].isin(normal_format)].copy()

In [6]:
# Extract Tambon: Stops at space or the 'อ.' prefix
df['tambon'] = df['location_name'].str.extract(r'ต\.\s*(.+?)(?=\s|อ\.|$)')

# Extract Amphoe: Stops at space or the 'จ.' prefix
df['amphoe'] = df['location_name'].str.extract(r'อ\.\s*(.+?)(?=\s|จ\.|$)')

# Extract Province: Stops at space or end of string
df['province'] = df['location_name'].str.extract(r'จ\.\s*(.+?)(?=\s|$)')

# Clean up any trailing dots (just in case)
for col in ['tambon', 'amphoe', 'province']:
    df[col] = df[col].str.strip(' .')

In [7]:
df['date'] = pd.to_datetime(df['log_datetime']).dt.date
df

,source_id,location_name,location_lat,location_lon,log_pm25,log_datetime,tambon,amphoe,province,date
0,4,ไนท์บาร์ซาร์ ต.ช้างม่อย อ.เมือง จ. เชียงใหม่,18.785477,98.999660,126.67,2026-04-01 00:00:00,ช้างม่อย,เมือง,เชียงใหม่,2026-04-01
1,4,ไนท์บาร์ซาร์ ต.ช้างม่อย อ.เมือง จ. เชียงใหม่,18.785477,98.999660,137.98,2026-04-01 01:00:00,ช้างม่อย,เมือง,เชียงใหม่,2026-04-01
2,4,ไนท์บาร์ซาร์ ต.ช้างม่อย อ.เมือง จ. เชียงใหม่,18.785477,98.999660,148.16,2026-04-01 02:00:00,ช้างม่อย,เมือง,เชียงใหม่,2026-04-01
3,4,ไนท์บาร์ซาร์ ต.ช้างม่อย อ.เมือง จ. เชียงใหม่,18.785477,98.999660,172.76,2026-04-01 03:00:00,ช้างม่อย,เมือง,เชียงใหม่,2026-04-01
4,4,ไนท์บาร์ซาร์ ต.ช้างม่อย อ.เมือง จ. เชียงใหม่,18.785477,98.999660,175.12,2026-04-01 04:00:00,ช้างม่อย,เมือง,เชียงใหม่,2026-04-01
...,...,...,...,...,...,...,...,...,...,...
438957,7171,วัดปงแท่น หมู่บ้านปงแท่น ต.จางเหนือ อ.แม่เมาะ ...,18.357949,99.892538,77.48,2026-04-30 19:00:00,จางเหนือ,แม่เมาะ,ลำปาง,2026-04-30
438958,7171,วัดปงแท่น หมู่บ้านปงแท่น ต.จางเหนือ อ.แม่เมาะ ...,18.357949,99.892538,81.48,2026-04-30 20:00:00,จางเหนือ,แม่เมาะ,ลำปาง,2026-04-30
438959,7171,วัดปงแท่น หมู่บ้านปงแท่น ต.จางเหนือ อ.แม่เมาะ ...,18.357949,99.892538,78.90,2026-04-30 21:00:00,จางเหนือ,แม่เมาะ,ลำปาง,2026-04-30
438960,7171,วัดปงแท่น หมู่บ้านปงแท่น ต.จางเหนือ อ.แม่เมาะ ...,18.357949,99.892538,78.90,2026-04-30 22:00:00,จางเหนือ,แม่เมาะ,ลำปาง,2026-04-30


In [8]:
print(len(list(df['province'].unique())))
print(list(df['province'].unique()))

65
['เชียงใหม่', 'แพร่', 'ลำพูน', 'แม่ฮ่องสอน', 'ตาก', 'ขอนแก่น', 'น่าน', 'เชียงราย', 'บึงกาฬ', 'สงขลา', 'พิษณุโลก', 'ลำปาง', 'กำแพงเพชร', 'ปทุมธานี', 'สุราษฎร์ธานี', 'นครราชสีมา', 'ราชบุรี', 'นครศรีธรรมราช', 'นนทบุรี', 'อุทัยธานี', 'พะเยา', 'สมุทรสาคร', 'มหาสารคาม', 'อุบลราชธานี', 'ศรีสะเกษ', 'นครนายก', 'พระนครศรีอยุธยา', 'ลพบุรี', 'อ่างทอง', 'สระบุรี', 'อุดรธานี', 'เลย', 'สตูล', 'ตรัง', 'ชัยนาท', 'พิจิตร', 'นครสวรรค์', 'ชัยภูมิ', 'ตราด', 'ระยอง', 'สมุทรปราการ', 'ชุมพร', 'พังงา', 'ภูเก็ต', 'กระบี่', 'ระนอง', 'สุพรรณบุรี', 'ชลบุรี', 'สุโขทัย', 'นครปฐม', 'ยะลา', 'ร้อยเอ็ด', 'สกลนคร', 'หนองคาย', 'สิงห์บุรี', 'สุรินทร์', 'เพชรบูรณ์', 'นครพนม', 'มุกดาหาร', 'ปัตตานี', 'กาญจนบุรี', 'สระแก้ว', 'กาฬสินธุ์', 'อุตรดิตถ์', 'บุรีรัมย์']


In [9]:
df['province'] = df['province'].replace({'ชียงใหม่': 'เชียงใหม่'})
print(len(list(df['province'].unique())))
print(list(df['province'].unique()))

65
['เชียงใหม่', 'แพร่', 'ลำพูน', 'แม่ฮ่องสอน', 'ตาก', 'ขอนแก่น', 'น่าน', 'เชียงราย', 'บึงกาฬ', 'สงขลา', 'พิษณุโลก', 'ลำปาง', 'กำแพงเพชร', 'ปทุมธานี', 'สุราษฎร์ธานี', 'นครราชสีมา', 'ราชบุรี', 'นครศรีธรรมราช', 'นนทบุรี', 'อุทัยธานี', 'พะเยา', 'สมุทรสาคร', 'มหาสารคาม', 'อุบลราชธานี', 'ศรีสะเกษ', 'นครนายก', 'พระนครศรีอยุธยา', 'ลพบุรี', 'อ่างทอง', 'สระบุรี', 'อุดรธานี', 'เลย', 'สตูล', 'ตรัง', 'ชัยนาท', 'พิจิตร', 'นครสวรรค์', 'ชัยภูมิ', 'ตราด', 'ระยอง', 'สมุทรปราการ', 'ชุมพร', 'พังงา', 'ภูเก็ต', 'กระบี่', 'ระนอง', 'สุพรรณบุรี', 'ชลบุรี', 'สุโขทัย', 'นครปฐม', 'ยะลา', 'ร้อยเอ็ด', 'สกลนคร', 'หนองคาย', 'สิงห์บุรี', 'สุรินทร์', 'เพชรบูรณ์', 'นครพนม', 'มุกดาหาร', 'ปัตตานี', 'กาญจนบุรี', 'สระแก้ว', 'กาฬสินธุ์', 'อุตรดิตถ์', 'บุรีรัมย์']


In [10]:
print(len(list(df['amphoe'].unique())))
print(list(df['amphoe'].unique()))

277
['เมือง', 'ฝาง', 'ร้องกวาง', 'สันป่าตอง', 'สารภี', 'ลี้', 'หางดง', 'แม่แจ่ม', 'เชียงดาว', 'ไชยปราการ', 'เมืองเชียงใหม่', 'ปาย', 'แม่ระมาด', 'หนองเรือ', 'สบเมย', 'นาหมื่น', 'แม่สาย', 'พร้าว', 'พญาเม็งราย', 'ป่าแดด', 'ปากคาด', 'เมืองลำพูน', 'หาดใหญ่', 'เนินมะปราง', 'แม่เมาะ', 'เมืองลำปาง', 'ดอยเต่า', 'ดอยสะเก็ด', 'สันกำแพง', 'สันทราย', 'คลองลาน', 'บึงกาฬ', 'ลำลูกกา', 'บ้านตาขุน', 'ปากช่อง', 'ดำเนินสะดวก', 'ทุ่งสง', 'บางบัวทอง', 'บ้านไร่', 'หนองฉาง', 'ทัพทัน', 'หนองขาหย่าง', 'บางกระทุ่ม', 'บางระกำ', 'อมก๋อย', 'เชียงคำ', 'มือง', 'ดอกคำใต้', 'ภูซาง', 'จุน', 'เชียงม่วน', 'แม่ทา', 'ทุ่งหัวช้าง', 'บ้านโฮ่ง', 'เชียงของ', 'พาน', 'เมืองสมุทรสาคร', 'ชื่นชม', 'กันทรารมย์', 'องครักษ์', 'บางใหญ่', 'บางปะอิน', 'นครหลวง', 'ลำสนธิ', 'โคกสำโรง', 'ไชโย', 'วิเศษชัยชาญ', 'มวกเหล็ก', 'แก่งคอย', 'พรเจริญ', 'ศรีวิไล', 'กุมภวาปี', 'ปากชม', 'เอราวัณ', 'ควนกาหลง', 'สิเกา', 'ย่านตาขาว', 'ละงู', 'ตะพานหิน', 'แม่ลาน้อย', 'ขุนยวม', 'ท่าตะโก', 'คอนสาร', 'คลองใหญ่', 'เขาสมิง', 'บ่อไร่', 'เกาะช้าง', 'บ้านฉาง', 'บางบ

In [11]:
df['amphoe'] = df['amphoe'].replace({'มือง': 'เมือง'})
print(len(list(df['amphoe'].unique())))
print(list(df['amphoe'].unique()))

276
['เมือง', 'ฝาง', 'ร้องกวาง', 'สันป่าตอง', 'สารภี', 'ลี้', 'หางดง', 'แม่แจ่ม', 'เชียงดาว', 'ไชยปราการ', 'เมืองเชียงใหม่', 'ปาย', 'แม่ระมาด', 'หนองเรือ', 'สบเมย', 'นาหมื่น', 'แม่สาย', 'พร้าว', 'พญาเม็งราย', 'ป่าแดด', 'ปากคาด', 'เมืองลำพูน', 'หาดใหญ่', 'เนินมะปราง', 'แม่เมาะ', 'เมืองลำปาง', 'ดอยเต่า', 'ดอยสะเก็ด', 'สันกำแพง', 'สันทราย', 'คลองลาน', 'บึงกาฬ', 'ลำลูกกา', 'บ้านตาขุน', 'ปากช่อง', 'ดำเนินสะดวก', 'ทุ่งสง', 'บางบัวทอง', 'บ้านไร่', 'หนองฉาง', 'ทัพทัน', 'หนองขาหย่าง', 'บางกระทุ่ม', 'บางระกำ', 'อมก๋อย', 'เชียงคำ', 'ดอกคำใต้', 'ภูซาง', 'จุน', 'เชียงม่วน', 'แม่ทา', 'ทุ่งหัวช้าง', 'บ้านโฮ่ง', 'เชียงของ', 'พาน', 'เมืองสมุทรสาคร', 'ชื่นชม', 'กันทรารมย์', 'องครักษ์', 'บางใหญ่', 'บางปะอิน', 'นครหลวง', 'ลำสนธิ', 'โคกสำโรง', 'ไชโย', 'วิเศษชัยชาญ', 'มวกเหล็ก', 'แก่งคอย', 'พรเจริญ', 'ศรีวิไล', 'กุมภวาปี', 'ปากชม', 'เอราวัณ', 'ควนกาหลง', 'สิเกา', 'ย่านตาขาว', 'ละงู', 'ตะพานหิน', 'แม่ลาน้อย', 'ขุนยวม', 'ท่าตะโก', 'คอนสาร', 'คลองใหญ่', 'เขาสมิง', 'บ่อไร่', 'เกาะช้าง', 'บ้านฉาง', 'บางบ่อ', 'บา

In [12]:
df_amphoe_scale = df[['province', 'amphoe', 'date', 'location_lat', 'location_lon', 'log_pm25']].dropna().groupby(by=['province', 'amphoe', 'date'], as_index=False).mean()
df_amphoe_scale

,province,amphoe,date,location_lat,location_lon,log_pm25
0,กระบี่,คลองท่อม,2026-04-01,7.952121,99.144181,6.978696
1,กระบี่,คลองท่อม,2026-04-02,7.952121,99.144181,9.960000
2,กระบี่,คลองท่อม,2026-04-03,7.952121,99.144181,15.111905
3,กระบี่,คลองท่อม,2026-04-04,7.952121,99.144181,23.923333
4,กระบี่,คลองท่อม,2026-04-05,7.952121,99.144181,26.953750
...,...,...,...,...,...,...
7774,แม่ฮ่องสอน,แม่สะเรียง,2026-04-26,18.062980,97.917651,35.004615
7775,แม่ฮ่องสอน,แม่สะเรียง,2026-04-27,18.062980,97.917651,36.059583
7776,แม่ฮ่องสอน,แม่สะเรียง,2026-04-28,18.062980,97.917651,30.323478
7777,แม่ฮ่องสอน,แม่สะเรียง,2026-04-29,18.062980,97.917651,32.819167


In [13]:
df_amphoe_scale.to_csv('../data/amphoe_pm.csv', index=False)